# Victorian Rental Rights RAG – Version 2 Evaluation

This notebook evaluates Version 2 of the Victorian Rental Rights RAG system using the same knowledge base, test collection, retrieval method and evaluation framework as Version 1.

Version 2 keeps the BM25 retrieval pipeline unchanged and introduces a revised generation approach designed to improve grounding, citation behaviour and handling of questions that are not supported by the knowledge base.

Keeping the remaining pipeline consistent allows Version 1 and Version 2 to be compared directly.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import math
import requests

from rank_bm25 import BM25Okapi

# Pipeline configuration
SYSTEM_VERSION = "V2"
OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"
TOP_K = 5

# Locate processed project data
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR / "data" / "processed"
elif (CURRENT_DIR.parent / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR.parent / "data" / "processed"
else:
    raise FileNotFoundError("Could not locate data/processed folder")

kb_df = pd.read_json(
    DATA_DIR / "rental_kb_chunks.jsonl",
    lines=True
)

test_df = pd.read_json(
    DATA_DIR / "rental_test_collection_v1.jsonl",
    lines=True
)

print(f"System version: {SYSTEM_VERSION}")
print(f"Model: {OLLAMA_MODEL}")
print(f"Top-k: {TOP_K}")
print(f"Knowledge-base chunks: {len(kb_df)}")
print(f"Test questions: {len(test_df)}")

System version: V2
Model: llama3.2:3b
Top-k: 5
Knowledge-base chunks: 32
Test questions: 15


## 1. Retrieval Pipeline

Build the BM25 retriever and retrieve the top-k knowledge-base chunks for every test question.

Retrieval metrics are calculated for Known and Inferred questions where ground-truth relevant chunks are available. Out-of-KB questions are still passed through retrieval, but are excluded from retrieval relevance metrics because they intentionally have no relevant chunks.

In [3]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


# Build BM25 index
tokenized_corpus = kb_df["text"].apply(tokenize).tolist()
bm25 = BM25Okapi(tokenized_corpus)


def retrieve_bm25(question, top_k=TOP_K):
    query_tokens = tokenize(question)
    scores = bm25.get_scores(query_tokens)

    results = kb_df.copy()
    results["bm25_score"] = scores

    return (
        results
        .sort_values("bm25_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )


def evaluate_retrieval(row, top_k=TOP_K):
    results = retrieve_bm25(row["question"], top_k)

    retrieved_ids = results["chunk_id"].tolist()
    retrieved_scores = results["bm25_score"].tolist()
    retrieved_contexts = results["text"].tolist()

    relevant_ids = set(row["relevant_chunk_ids"])

    # Out-of-KB questions have no relevant chunks
    if not relevant_ids:
        return pd.Series({
            "retrieved_chunk_ids": retrieved_ids,
            "retrieved_scores": retrieved_scores,
            "retrieved_contexts": retrieved_contexts,
            "first_relevant_rank": np.nan,
            "recall_at_k": np.nan,
            "mrr_at_k": np.nan,
            "ndcg_at_k": np.nan
        })

    matched = [
        chunk_id for chunk_id in retrieved_ids
        if chunk_id in relevant_ids
    ]

    recall = len(matched) / len(relevant_ids)

    first_rank = next(
        (
            rank
            for rank, chunk_id in enumerate(retrieved_ids, start=1)
            if chunk_id in relevant_ids
        ),
        None
    )

    mrr = 1 / first_rank if first_rank else 0

    dcg = sum(
        1 / math.log2(rank + 1)
        for rank, chunk_id in enumerate(retrieved_ids, start=1)
        if chunk_id in relevant_ids
    )

    ideal_relevant = min(len(relevant_ids), top_k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    return pd.Series({
        "retrieved_chunk_ids": retrieved_ids,
        "retrieved_scores": retrieved_scores,
        "retrieved_contexts": retrieved_contexts,
        "first_relevant_rank": first_rank,
        "recall_at_k": recall,
        "mrr_at_k": mrr,
        "ndcg_at_k": ndcg
    })


retrieval_results = test_df.apply(
    evaluate_retrieval,
    axis=1
)

results_df = pd.concat(
    [test_df.copy(), retrieval_results],
    axis=1
)

display(
    results_df[
        [
            "question_id",
            "question_type",
            "retrieved_chunk_ids",
            "first_relevant_rank",
            "recall_at_k",
            "mrr_at_k",
            "ndcg_at_k"
        ]
    ]
)

,question_id,question_type,retrieved_chunk_ids,first_relevant_rank,recall_at_k,mrr_at_k,ndcg_at_k
0,K01,Known,"[RG_P35, RG_P38, RG_P04, RG_P23, RG_P07]",5.0,1.00,0.2,0.386853
1,K02,Known,"[RG_P08, RG_P21, MS_P01, RG_P12, RG_P13]",1.0,1.00,1.0,1.000000
2,K03,Known,"[RG_P13, MS_P01, RG_P39, RG_P17, RG_P04]",1.0,1.00,1.0,1.000000
3,K04,Known,"[RG_P23, RG_P17, RG_P19, RG_P34, RG_P36]",2.0,1.00,0.5,0.693426
4,K05,Known,"[RG_P21, RG_P22, RG_P17, RG_P23, RG_P08]",1.0,1.00,1.0,1.000000
5,K06,Known,"[RG_P23, RG_P22, RG_P17, RG_P24, RG_P18]",2.0,1.00,0.5,0.630930
6,K07,Known,"[RG_P24, RG_P25, RG_P26, RG_P15, MS_P01]",1.0,1.00,1.0,1.000000
7,K08,Known,"[RG_P28, RG_P23, RG_P22, RG_P29, RG_P34]",1.0,1.00,1.0,1.000000
8,I01,Inferred,"[RG_P25, RG_P26, MS_P01, RG_P23, RG_P13]",1.0,0.75,1.0,0.736590
9,I02,Inferred,"[RG_P23, RG_P22, RG_P17, RG_P18, RG_P25]",1.0,1.00,1.0,1.000000


## 2. Version 2 Answer Generation

Version 2 keeps the same retrieved top-five chunks but changes how the language model is instructed to use them.

The revised prompt is designed to improve three weaknesses identified in Version 1:

- answering questions when relevant evidence is present anywhere in the retrieved context
- citing the evidence used for factual claims
- abstaining when none of the retrieved chunks contain enough information to answer

The model is also instructed to cover all parts of the question that are supported by the retrieved evidence and avoid filling gaps using outside knowledge.

In [4]:
def build_context(row):
    context_parts = []

    for chunk_id, text in zip(
        row["retrieved_chunk_ids"],
        row["retrieved_contexts"]
    ):
        chunk = kb_df.loc[
            kb_df["chunk_id"] == chunk_id
        ].iloc[0]

        context_parts.append(
            f"[{chunk_id}] "
            f"{chunk['document_name']} - Page {chunk['page']}\n"
            f"{text}"
        )

    return "\n\n---\n\n".join(context_parts)


def generate_answer(question, context):
    prompt = f"""
You are answering a question about Victorian rental rights using retrieved source material.

Read ALL of the retrieved chunks before answering. Relevant evidence may appear anywhere in the context.

Rules:
- Answer using only information explicitly supported by the retrieved context.
- If any retrieved chunk directly supports the answer, use that evidence and answer the question.
- Do not refuse simply because some retrieved chunks are irrelevant.
- Answer all parts of the question that are supported by the retrieved evidence.
- Do not invent, assume or infer rules that are not stated in the context.
- Cite the supporting chunk ID after each factual statement, for example [RG_P24].
- Only cite chunks that actually support the statement.
- If none of the retrieved chunks contain enough information to answer the question, respond exactly:
  "The retrieved information does not contain enough information to answer this question."

RETRIEVED CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0
            }
        }
    )

    response.raise_for_status()

    return response.json()["response"].strip()


# Build retrieved context
results_df["retrieved_context"] = results_df.apply(
    build_context,
    axis=1
)

# Generate Version 2 answers
results_df["generated_answer"] = results_df.apply(
    lambda row: generate_answer(
        row["question"],
        row["retrieved_context"]
    ),
    axis=1
)

display(
    results_df[
        [
            "question_id",
            "question_type",
            "generated_answer"
        ]
    ]
)

,question_id,question_type,generated_answer
0,K01,Known,[RG_P07] Renters Guide - Page 7\n\nThe rental ...
1,K02,Known,"According to [RG_P21], a rental provider is al..."
2,K03,Known,"Yes, a Victorian rental property must have a f..."
3,K04,Known,"According to [RG_P17], you must give one copy ..."
4,K05,Known,"According to [RG_P21], the most you can be ask..."
5,K06,Known,"According to [RG_P24], your rental provider mu..."
6,K07,Known,The retrieved context does not provide informa...
7,K08,Known,"According to [RG_P28], if your rental provider..."
8,I01,Inferred,"According to [RG_P13], a rental property must ..."
9,I02,Inferred,"According to [RG_P23], your rental provider mu..."


## 3. Automatic Output Checks

Some RAG behaviours can be evaluated directly without using another language model as a judge.

This stage checks whether the generated answer contains citations, whether those citations refer to chunks that were actually retrieved, and whether the model abstained because it could not find enough information. These objective checks are kept separate from later semantic measures such as correctness, completeness and faithfulness.

In [5]:
# Extract cited chunk IDs from generated answers
def extract_citations(answer):
    return list(dict.fromkeys(
        re.findall(r"\b(?:RG|MS)_P\d{2}\b", answer)
    ))


# Detect common abstention responses
def detect_abstention(answer):
    text = answer.lower()

    abstention_phrases = [
        "does not contain enough information",
        "not enough information",
        "no mention of",
        "not mentioned in",
        "cannot determine from",
        "cannot be determined from",
        "context does not provide",
        "context does not contain"
    ]

    return int(any(
        phrase in text
        for phrase in abstention_phrases
    ))


results_df["cited_chunk_ids"] = (
    results_df["generated_answer"]
    .apply(extract_citations)
)

results_df["citation_present"] = (
    results_df["cited_chunk_ids"]
    .apply(lambda x: int(len(x) > 0))
)

# Check that cited chunks were actually supplied to the model
results_df["citations_from_retrieved"] = results_df.apply(
    lambda row:
        int(
            all(
                citation in row["retrieved_chunk_ids"]
                for citation in row["cited_chunk_ids"]
            )
        )
        if row["cited_chunk_ids"]
        else np.nan,
    axis=1
)

results_df["abstained"] = (
    results_df["generated_answer"]
    .apply(detect_abstention)
)

# Ground-truth expectation for abstention
results_df["should_abstain"] = (
    results_df["question_type"] == "Out-of-KB"
).astype(int)

results_df["abstention_correct"] = (
    results_df["abstained"]
    == results_df["should_abstain"]
).astype(int)


display(
    results_df[
        [
            "question_id",
            "question_type",
            "cited_chunk_ids",
            "citation_present",
            "citations_from_retrieved",
            "abstained",
            "should_abstain",
            "abstention_correct"
        ]
    ]
)

,question_id,question_type,cited_chunk_ids,citation_present,citations_from_retrieved,abstained,should_abstain,abstention_correct
0,K01,Known,[RG_P07],1,1.0,0,0,1
1,K02,Known,[RG_P21],1,1.0,0,0,1
2,K03,Known,"[MS_P01, RG_P39]",1,1.0,0,0,1
3,K04,Known,[RG_P17],1,1.0,0,0,1
4,K05,Known,[RG_P21],1,1.0,0,0,1
5,K06,Known,[RG_P24],1,1.0,0,0,1
6,K07,Known,[],0,NaN,1,0,0
7,K08,Known,[RG_P28],1,1.0,0,0,1
8,I01,Inferred,[RG_P13],1,1.0,0,0,1
9,I02,Inferred,[RG_P23],1,1.0,0,0,1


### Automatic Evaluation Summary

Citation and abstention behaviour can be measured directly from the generated outputs. These checks provide objective system-level metrics before evaluating the meaning and factual support of individual answers.

In [8]:
# Summarise deterministic generation checks

answers_with_citations = results_df[
    "citation_present"
].mean()

valid_citations = results_df.loc[
    results_df["citation_present"] == 1,
    "citations_from_retrieved"
].mean()

out_of_kb = results_df[
    results_df["question_type"] == "Out-of-KB"
]

out_of_kb_abstention = out_of_kb[
    "abstention_correct"
].mean()

automatic_summary = pd.DataFrame({
    "metric": [
        "Citation presence",
        "Citations from retrieved context",
        "Out-of-KB abstention accuracy"
    ],
    "score": [
        answers_with_citations,
        valid_citations,
        out_of_kb_abstention
    ]
})

automatic_summary["score"] = (
    automatic_summary["score"]
    .round(3)
)

display(automatic_summary)

,metric,score
0,Citation presence,0.733
1,Citations from retrieved context,1.000
2,Out-of-KB abstention accuracy,1.000


## 4. Gold-Fact Evaluation

Version 2 is evaluated against the same gold facts used for Version 1.

Each expected answer is broken into individual factual requirements, allowing completeness to be measured as the proportion of required information included in the generated answer.

Keeping the gold facts unchanged ensures that Version 1 and Version 2 are evaluated against the same reference standard.

In [9]:
# Required facts for each supported test question

gold_facts = {
    "K01": [
        "A rental provider or agent cannot ask whether the applicant has taken legal action or had a dispute with a previous rental provider."
    ],

    "K02": [
        "Rental providers or agents cannot accept an offer above the advertised rent."
    ],

    "K03": [
        "The property must have a fixed heater in good working order in the main living area.",
        "For rental agreements starting from 29 March 2023, the heater must be energy efficient."
    ],

    "K04": [
        "The completed and signed condition report must be returned within 5 business days of moving in."
    ],

    "K05": [
        "If rent is paid weekly, no more than 2 weeks rent can be requested in advance.",
        "If rent is paid monthly and weekly rent is $900 or less, no more than one month rent can be requested in advance.",
        "Different rules apply when weekly rent is above $900."
    ],

    "K06": [
        "A Notice of rent increase must be provided at least 90 days before the increase takes effect."
    ],

    "K07": [
        "A blocked or broken toilet system is an urgent repair.",
        "Urgent repairs must be dealt with immediately."
    ],

    "K08": [
        "A rental provider who wants to refuse a pet request must apply to VCAT within 14 days.",
        "VCAT decides whether refusing consent is reasonable."
    ],

    "I01": [
        "A rental property must have a working fixed heater in the main living area.",
        "Failure to meet rental minimum standards is considered an urgent repair.",
        "The rental provider must arrange an urgent repair immediately.",
        "If the provider does not respond, the renter can arrange the urgent repair if it costs no more than $2500.",
        "The rental provider must reimburse the renter within 7 days.",
        "If reimbursement is not provided, the renter can apply to RDRV."
    ],

    "I02": [
        "The renter must receive at least 90 days notice of a rent increase.",
        "The renter can request a free rent assessment from Consumer Affairs Victoria.",
        "The assessment must be requested within 30 days of receiving the rent increase notice.",
        "If the dispute remains unresolved, the renter can apply to RDRV or VCAT."
    ],

    "I03": [
        "The original condition report can show that the damage existed when the renter moved in.",
        "Photos can provide evidence of pre-existing damage.",
        "The exit condition report can be compared with the original condition report.",
        "The rental provider cannot claim the bond for fair wear and tear.",
        "A disputed bond claim can be taken through RDRV."
    ],

    "I04": [
        "A rental provider cannot evict a renter because they exercised or intended to exercise their rental rights.",
        "The rental provider needs a valid reason to end the agreement.",
        "The rental provider must provide the correct written Notice to vacate.",
        "The required notice period depends on the reason.",
        "In most instances the notice period is 90 days."
    ],

    "O01": [],
    "O02": [],
    "O03": []
}

results_df["gold_facts"] = (
    results_df["question_id"]
    .map(gold_facts)
)

results_df["gold_fact_count"] = (
    results_df["gold_facts"]
    .apply(len)
)

display(
    results_df[
        [
            "question_id",
            "question_type",
            "gold_fact_count",
            "gold_facts"
        ]
    ]
)

,question_id,question_type,gold_fact_count,gold_facts
0,K01,Known,1,[A rental provider or agent cannot ask whether...
1,K02,Known,1,[Rental providers or agents cannot accept an o...
2,K03,Known,2,[The property must have a fixed heater in good...
3,K04,Known,1,[The completed and signed condition report mus...
4,K05,Known,3,"[If rent is paid weekly, no more than 2 weeks ..."
5,K06,Known,1,[A Notice of rent increase must be provided at...
6,K07,Known,2,[A blocked or broken toilet system is an urgen...
7,K08,Known,2,[A rental provider who wants to refuse a pet r...
8,I01,Inferred,6,[A rental property must have a working fixed h...
9,I02,Inferred,4,[The renter must receive at least 90 days noti...


### Human Gold-Fact Evaluation

Each Version 2 answer is reviewed against the same gold facts used for Version 1.

A gold fact is marked as `1` when the required information is clearly included in the generated answer and `0` when it is missing or contradicted.

The resulting completeness scores can therefore be compared directly between the two system versions.

In [10]:
# Create one evaluation row for each required gold fact

fact_rows = []

for _, row in results_df.iterrows():
    for fact_number, fact in enumerate(
        row["gold_facts"],
        start=1
    ):
        fact_rows.append({
            "question_id": row["question_id"],
            "question_type": row["question_type"],
            "fact_number": fact_number,
            "gold_fact": fact,
            "generated_answer": row["generated_answer"],
            "covered": pd.NA,
            "review_notes": ""
        })

fact_eval_df = pd.DataFrame(fact_rows)

print("Total gold facts:", len(fact_eval_df))

display(fact_eval_df)

Total gold facts: 33


,question_id,question_type,fact_number,gold_fact,generated_answer,covered,review_notes
0,K01,Known,1,A rental provider or agent cannot ask whether ...,[RG_P07] Renters Guide - Page 7\n\nThe rental ...,<NA>,
1,K02,Known,1,Rental providers or agents cannot accept an of...,"According to [RG_P21], a rental provider is al...",<NA>,
2,K03,Known,1,The property must have a fixed heater in good ...,"Yes, a Victorian rental property must have a f...",<NA>,
3,K03,Known,2,For rental agreements starting from 29 March 2...,"Yes, a Victorian rental property must have a f...",<NA>,
4,K04,Known,1,The completed and signed condition report must...,"According to [RG_P17], you must give one copy ...",<NA>,
5,K05,Known,1,"If rent is paid weekly, no more than 2 weeks r...","According to [RG_P21], the most you can be ask...",<NA>,
6,K05,Known,2,If rent is paid monthly and weekly rent is $90...,"According to [RG_P21], the most you can be ask...",<NA>,
7,K05,Known,3,Different rules apply when weekly rent is abov...,"According to [RG_P21], the most you can be ask...",<NA>,
8,K06,Known,1,A Notice of rent increase must be provided at ...,"According to [RG_P24], your rental provider mu...",<NA>,
9,K07,Known,1,A blocked or broken toilet system is an urgent...,The retrieved context does not provide informa...,<NA>,


In [11]:
for _, row in results_df[
    results_df["gold_fact_count"] > 0
].iterrows():

    print("=" * 80)
    print(f"{row['question_id']} - {row['question_type']}")
    print("\nANSWER:")
    print(row["generated_answer"])

    print("\nREQUIRED GOLD FACTS:")
    for i, fact in enumerate(
        row["gold_facts"],
        start=1
    ):
        print(f"{i}. {fact}")

    print()

K01 - Known

ANSWER:
[RG_P07] Renters Guide - Page 7

The rental provider or agent cannot ask you for the following information in your application:

• whether you’ve taken legal action or had a dispute with a previous rental provider.

This is explicitly stated in the text.

REQUIRED GOLD FACTS:
1. A rental provider or agent cannot ask whether the applicant has taken legal action or had a dispute with a previous rental provider.

K02 - Known

ANSWER:
According to [RG_P21], a rental provider is allowed to invite a tenant to pay more than one month's rent in advance if the weekly rent is above $900.

REQUIRED GOLD FACTS:
1. Rental providers or agents cannot accept an offer above the advertised rent.

K03 - Known

ANSWER:
Yes, a Victorian rental property must have a fixed heater in good working order in the main living area. [MS_P01: Heating, RG_P39: Renters Guide - Page 39]

According to the standards, a fixed heater in the main living area must be in good working order. For rental agre

### Record Human Fact Judgements

Each gold fact is manually reviewed against the generated answer. A value of `1` means the required information is clearly present, while `0` means it is missing or contradicted.

These human labels provide the reference completeness evaluation for the baseline system.

In [12]:
# Human-reviewed Version 2 gold fact coverage
# Format: (question_id, fact_number): covered

fact_coverage = {
    ("K01", 1): 1,

    ("K02", 1): 0,

    ("K03", 1): 1,
    ("K03", 2): 1,

    ("K04", 1): 1,

    ("K05", 1): 1,
    ("K05", 2): 1,
    ("K05", 3): 1,

    ("K06", 1): 1,

    ("K07", 1): 0,
    ("K07", 2): 0,

    ("K08", 1): 1,
    ("K08", 2): 0,

    ("I01", 1): 1,
    ("I01", 2): 0,
    ("I01", 3): 0,
    ("I01", 4): 1,
    ("I01", 5): 0,
    ("I01", 6): 0,

    ("I02", 1): 1,
    ("I02", 2): 0,
    ("I02", 3): 1,
    ("I02", 4): 0,

    ("I03", 1): 0,
    ("I03", 2): 1,
    ("I03", 3): 0,
    ("I03", 4): 0,
    ("I03", 5): 1,

    ("I04", 1): 1,
    ("I04", 2): 1,
    ("I04", 3): 0,
    ("I04", 4): 0,
    ("I04", 5): 0
}


fact_eval_df["covered"] = fact_eval_df.apply(
    lambda row: fact_coverage.get(
        (row["question_id"], row["fact_number"]),
        pd.NA
    ),
    axis=1
)

# Check that every gold fact has been reviewed
print("Total gold facts:", len(fact_eval_df))
print(
    "Reviewed facts:",
    fact_eval_df["covered"].notna().sum()
)
print(
    "Missing judgements:",
    fact_eval_df["covered"].isna().sum()
)

display(
    fact_eval_df[
        [
            "question_id",
            "fact_number",
            "gold_fact",
            "covered"
        ]
    ]
)

Total gold facts: 33
Reviewed facts: 33
Missing judgements: 0


,question_id,fact_number,gold_fact,covered
0,K01,1,A rental provider or agent cannot ask whether ...,1
1,K02,1,Rental providers or agents cannot accept an of...,0
2,K03,1,The property must have a fixed heater in good ...,1
3,K03,2,For rental agreements starting from 29 March 2...,1
4,K04,1,The completed and signed condition report must...,1
5,K05,1,"If rent is paid weekly, no more than 2 weeks r...",1
6,K05,2,If rent is paid monthly and weekly rent is $90...,1
7,K05,3,Different rules apply when weekly rent is abov...,1
8,K06,1,A Notice of rent increase must be provided at ...,1
9,K07,1,A blocked or broken toilet system is an urgent...,0


### Version 2 Completeness Scores

Gold-fact judgements are aggregated into a completeness score for each supported question.

The score is the proportion of required gold facts included in the generated answer. The same calculation used for Version 1 is retained so the two system versions can be compared directly.

In [13]:
# Calculate completeness for each supported question

question_completeness = (
    fact_eval_df
    .groupby(
        ["question_id", "question_type"]
    )
    .agg(
        facts_covered=("covered", "sum"),
        total_facts=("covered", "count")
    )
    .reset_index()
)

question_completeness["completeness_score"] = (
    question_completeness["facts_covered"]
    / question_completeness["total_facts"]
)

# Add question-level completeness back to main results
completeness_map = (
    question_completeness
    .set_index("question_id")["completeness_score"]
)

results_df["completeness_score"] = (
    results_df["question_id"]
    .map(completeness_map)
)

# Mean completeness by question type
completeness_by_type = (
    question_completeness
    .groupby("question_type")["completeness_score"]
    .mean()
    .round(3)
    .reset_index()
)

# Overall completeness measures
overall_question_completeness = (
    question_completeness["completeness_score"].mean()
)

overall_fact_coverage = (
    fact_eval_df["covered"].sum()
    / len(fact_eval_df)
)

display(question_completeness)

display(completeness_by_type)

print(
    "Overall mean question completeness:",
    round(overall_question_completeness, 3)
)

print(
    "Overall gold-fact coverage:",
    round(overall_fact_coverage, 3)
)

,question_id,question_type,facts_covered,total_facts,completeness_score
0,I01,Inferred,2,6,0.333333
1,I02,Inferred,2,4,0.500000
2,I03,Inferred,2,5,0.400000
3,I04,Inferred,2,5,0.400000
4,K01,Known,1,1,1.000000
5,K02,Known,0,1,0.000000
6,K03,Known,2,2,1.000000
7,K04,Known,1,1,1.000000
8,K05,Known,3,3,1.000000
9,K06,Known,1,1,1.000000


,question_type,completeness_score
0,Inferred,0.408
1,Known,0.688


Overall mean question completeness: 0.594
Overall gold-fact coverage: 0.515


## 5. Claim-Level Faithfulness Evaluation

Faithfulness measures whether the factual claims made by Version 2 are supported by the retrieved context that was supplied to the model.

Each generated answer is broken into individual factual claims using the same claim-level approach as Version 1. Each claim is then manually reviewed and marked as supported (`1`) or unsupported (`0`).

This provides two measures:

- **Faithfulness:** proportion of generated factual claims supported by retrieved evidence.
- **Unsupported claim rate:** proportion of generated factual claims not supported by retrieved evidence.

Abstention responses are not treated as factual rental-rights claims. Their performance is evaluated separately through the abstention metrics.

In [14]:
# Manually identified factual claims from Version 2 answers

generated_claims = {
    "K01": [
        "A rental provider or agent cannot ask whether an applicant has taken legal action or had a dispute with a previous rental provider."
    ],

    "K02": [
        "If weekly rent is above $900, a rental provider can invite a renter to pay more than one month's rent in advance."
    ],

    "K03": [
        "A Victorian rental property must have a fixed heater in good working order in the main living area.",
        "For rental agreements starting from 29 March 2023, the fixed heater in the main living area must be energy efficient."
    ],

    "K04": [
        "The completed and signed condition report must be returned within 5 business days of moving in."
    ],

    "K05": [
        "If rent is paid weekly, up to 2 weeks rent can be requested in advance.",
        "If rent is paid monthly and weekly rent is $900 or less, up to one month rent can be requested in advance.",
        "If weekly rent is above $900, more than one month's rent can be requested in advance."
    ],

    "K06": [
        "A rental provider must give at least 90 days notice before a rent increase."
    ],

    # Incorrect abstention, but no factual rental-rights claim was generated
    "K07": [],

    "K08": [
        "A rental provider who wants to refuse a pet request must apply to VCAT within 14 days."
    ],

    "I01": [
        "A rental property must have a fixed heater in good working order in the main living area.",
        "A non-working fixed heater is considered a non-urgent repair.",
        "If the rental provider does not respond, the renter can arrange and pay for the repair if it costs no more than $2500.",
        "The renter can use the Notice to rental provider of rented premises form to request reimbursement.",
        "Receipts, invoices or other proof of repairs are needed when requesting reimbursement."
    ],

    "I02": [
        "A rental provider must use the Notice of rent increase form and provide at least 90 days notice before increasing the rent.",
        "A renter can ask Consumer Affairs Victoria to investigate if they believe a rent increase is too high.",
        "The renter must contact Consumer Affairs Victoria within 30 days of receiving the rent increase notice.",
        "The renter can use the Request for rental assessment form to request a rent assessment."
    ],

    "I03": [
        "A renter can provide evidence when disputing a rental provider's bond claim.",
        "RDRV can review the bond dispute, discuss it with both parties and try to help reach a fair outcome.",
        "When moving out, the property must be reasonably clean and in the same condition as when the renter moved in, allowing for fair wear and tear.",
        "Pre-existing damage can be used as a reason to dispute responsibility for the damage.",
        "Photos or other documentation can be used as evidence that damage existed before the renter moved in.",
        "Rental agreements signed on or after 29 March 2021 can have additional professional cleaning requirements."
    ],

    "I04": [
        "A rental provider cannot end a rental agreement without a valid reason, including at the end of a fixed-term agreement.",
        "Valid reasons can include the provider or family moving in, major renovations or demolition.",
        "A rental provider cannot evict a renter for using or intending to use their rental rights."
    ],

    # Correct abstentions contain no rental-rights factual claims
    "O01": [],
    "O02": [],
    "O03": []
}


claim_rows = []

for _, row in results_df.iterrows():
    for claim_number, claim in enumerate(
        generated_claims[row["question_id"]],
        start=1
    ):
        claim_rows.append({
            "question_id": row["question_id"],
            "question_type": row["question_type"],
            "claim_number": claim_number,
            "claim": claim,
            "retrieved_chunk_ids": row["retrieved_chunk_ids"],
            "supported": pd.NA,
            "supporting_chunk_ids": [],
            "review_notes": ""
        })

claim_eval_df = pd.DataFrame(claim_rows)

print(
    "Total generated factual claims:",
    len(claim_eval_df)
)

display(claim_eval_df)

Total generated factual claims: 28


,question_id,question_type,claim_number,claim,retrieved_chunk_ids,supported,supporting_chunk_ids,review_notes
0,K01,Known,1,A rental provider or agent cannot ask whether ...,"[RG_P35, RG_P38, RG_P04, RG_P23, RG_P07]",<NA>,[],
1,K02,Known,1,"If weekly rent is above $900, a rental provide...","[RG_P08, RG_P21, MS_P01, RG_P12, RG_P13]",<NA>,[],
2,K03,Known,1,A Victorian rental property must have a fixed ...,"[RG_P13, MS_P01, RG_P39, RG_P17, RG_P04]",<NA>,[],
3,K03,Known,2,For rental agreements starting from 29 March 2...,"[RG_P13, MS_P01, RG_P39, RG_P17, RG_P04]",<NA>,[],
4,K04,Known,1,The completed and signed condition report must...,"[RG_P23, RG_P17, RG_P19, RG_P34, RG_P36]",<NA>,[],
5,K05,Known,1,"If rent is paid weekly, up to 2 weeks rent can...","[RG_P21, RG_P22, RG_P17, RG_P23, RG_P08]",<NA>,[],
6,K05,Known,2,If rent is paid monthly and weekly rent is $90...,"[RG_P21, RG_P22, RG_P17, RG_P23, RG_P08]",<NA>,[],
7,K05,Known,3,"If weekly rent is above $900, more than one mo...","[RG_P21, RG_P22, RG_P17, RG_P23, RG_P08]",<NA>,[],
8,K06,Known,1,A rental provider must give at least 90 days n...,"[RG_P23, RG_P22, RG_P17, RG_P24, RG_P18]",<NA>,[],
9,K08,Known,1,A rental provider who wants to refuse a pet re...,"[RG_P28, RG_P23, RG_P22, RG_P29, RG_P34]",<NA>,[],


### Human Claim Support Review

Each Version 2 factual claim is reviewed against the retrieved context that was actually supplied to the model.

A claim is marked as:

- `1` when the retrieved evidence clearly supports the claim
- `0` when the claim is unsupported or contradicted by the retrieved evidence

This is kept separate from answer correctness. A claim can be supported by the retrieved context but still fail to answer the original question correctly. Citation accuracy is also evaluated separately in the next section.

In [15]:
# Human-reviewed Version 2 claim support
# Format:
# (question_id, claim_number): (supported, supporting_chunk_ids, review_note)

claim_support = {
    ("K01", 1): (
        1,
        ["RG_P07"],
        "Supported by the application rules."
    ),

    # The answer does not address rental bidding, but this specific claim
    # is supported by the retrieved rent-in-advance rules.
    ("K02", 1): (
        1,
        ["RG_P21"],
        "The claim is supported by the rent-in-advance rules, although it does not answer the original rental-bidding question."
    ),

    ("K03", 1): (
        1,
        ["RG_P13", "MS_P01"],
        "Supported by the minimum heating requirements."
    ),
    ("K03", 2): (
        1,
        ["RG_P13", "MS_P01"],
        "Supported by the energy-efficiency requirement for agreements starting from 29 March 2023."
    ),

    ("K04", 1): (
        1,
        ["RG_P17"],
        "Supported by the condition report requirements."
    ),

    ("K05", 1): (
        1,
        ["RG_P21"],
        "Supported by the weekly rent-in-advance rule."
    ),
    ("K05", 2): (
        1,
        ["RG_P21"],
        "Supported by the monthly rent-in-advance rule."
    ),
    ("K05", 3): (
        1,
        ["RG_P21"],
        "Supported by the rule for weekly rent above $900."
    ),

    ("K06", 1): (
        1,
        ["RG_P22", "RG_P23"],
        "The 90-day rent increase notice requirement is supported by the retrieved evidence."
    ),

    ("K08", 1): (
        1,
        ["RG_P28"],
        "Supported by the pet refusal process."
    ),

    ("I01", 1): (
        1,
        ["RG_P13", "MS_P01"],
        "Supported by the minimum heating requirements."
    ),
    ("I01", 2): (
        0,
        [],
        "The retrieved evidence does not support classifying a non-working fixed heater as a non-urgent repair."
    ),
    ("I01", 3): (
        0,
        [],
        "The $2500 self-arranged repair process applies to urgent repairs, but the answer applies it after incorrectly classifying the heater issue as non-urgent."
    ),
    ("I01", 4): (
        1,
        ["RG_P25"],
        "The retrieved urgent-repair process supports using the relevant notice form to request reimbursement."
    ),
    ("I01", 5): (
        1,
        ["RG_P25"],
        "The retrieved repair process supports providing receipts, invoices or other evidence of the repair."
    ),

    ("I02", 1): (
        1,
        ["RG_P22", "RG_P23"],
        "Supported by the rent increase notice requirements."
    ),
    ("I02", 2): (
        1,
        ["RG_P23"],
        "Supported by the Consumer Affairs Victoria rent assessment process."
    ),
    ("I02", 3): (
        1,
        ["RG_P23"],
        "Supported by the 30-day assessment deadline."
    ),
    ("I02", 4): (
        1,
        ["RG_P23"],
        "Supported by the rent assessment process."
    ),

    ("I03", 1): (
        1,
        ["RG_P35"],
        "Supported by the bond dispute process."
    ),
    ("I03", 2): (
        1,
        ["RG_P35"],
        "Supported by the RDRV bond dispute process."
    ),
    ("I03", 3): (
        1,
        ["RG_P34"],
        "Supported by the moving-out and fair wear and tear requirements."
    ),
    ("I03", 4): (
        1,
        ["RG_P17", "RG_P34"],
        "The retrieved condition-report and moving-out evidence supports using pre-existing damage when disputing responsibility."
    ),
    ("I03", 5): (
        1,
        ["RG_P17"],
        "Photos can be used to document the property's condition."
    ),
    ("I03", 6): (
        1,
        ["RG_P34"],
        "Supported by the professional cleaning requirements."
    ),

    ("I04", 1): (
        1,
        ["RG_P32"],
        "Supported by the rules for ending a rental agreement."
    ),
    ("I04", 2): (
        1,
        ["RG_P32"],
        "Supported by the listed valid reasons for issuing a Notice to vacate."
    ),
    ("I04", 3): (
        1,
        ["RG_P33"],
        "Supported by the protection against eviction for exercising rental rights."
    )
}


# Apply the human judgements
for index, row in claim_eval_df.iterrows():
    key = (
        row["question_id"],
        row["claim_number"]
    )

    if key in claim_support:
        supported, chunks, note = claim_support[key]

        claim_eval_df.at[index, "supported"] = supported
        claim_eval_df.at[index, "supporting_chunk_ids"] = chunks
        claim_eval_df.at[index, "review_notes"] = note


# Check that all Version 2 claims were reviewed
print(
    "Total generated claims:",
    len(claim_eval_df)
)

print(
    "Reviewed claims:",
    claim_eval_df["supported"].notna().sum()
)

print(
    "Missing judgements:",
    claim_eval_df["supported"].isna().sum()
)

display(
    claim_eval_df[
        [
            "question_id",
            "claim_number",
            "claim",
            "supported",
            "supporting_chunk_ids",
            "review_notes"
        ]
    ]
)


Total generated claims: 28
Reviewed claims: 28
Missing judgements: 0


,question_id,claim_number,claim,supported,supporting_chunk_ids,review_notes
0,K01,1,A rental provider or agent cannot ask whether ...,1,[RG_P07],Supported by the application rules.
1,K02,1,"If weekly rent is above $900, a rental provide...",1,[RG_P21],The claim is supported by the rent-in-advance ...
2,K03,1,A Victorian rental property must have a fixed ...,1,"[RG_P13, MS_P01]",Supported by the minimum heating requirements.
3,K03,2,For rental agreements starting from 29 March 2...,1,"[RG_P13, MS_P01]",Supported by the energy-efficiency requirement...
4,K04,1,The completed and signed condition report must...,1,[RG_P17],Supported by the condition report requirements.
5,K05,1,"If rent is paid weekly, up to 2 weeks rent can...",1,[RG_P21],Supported by the weekly rent-in-advance rule.
6,K05,2,If rent is paid monthly and weekly rent is $90...,1,[RG_P21],Supported by the monthly rent-in-advance rule.
7,K05,3,"If weekly rent is above $900, more than one mo...",1,[RG_P21],Supported by the rule for weekly rent above $900.
8,K06,1,A rental provider must give at least 90 days n...,1,"[RG_P22, RG_P23]",The 90-day rent increase notice requirement is...
9,K08,1,A rental provider who wants to refuse a pet re...,1,[RG_P28],Supported by the pet refusal process.


### Version 2 Faithfulness Scores

The human claim judgements are aggregated to measure how consistently Version 2 stays grounded in the retrieved evidence.

Faithfulness is calculated as the proportion of generated factual claims supported by the retrieved context. The unsupported claim rate reports the corresponding proportion of claims that are not supported.

The same calculation used for Version 1 is retained for the A/B comparison.

In [16]:
# Calculate claim-level faithfulness

claim_eval_df["supported"] = pd.to_numeric(
    claim_eval_df["supported"]
)

# Question-level faithfulness
question_faithfulness = (
    claim_eval_df
    .groupby(
        ["question_id", "question_type"]
    )
    .agg(
        supported_claims=("supported", "sum"),
        total_claims=("supported", "count")
    )
    .reset_index()
)

question_faithfulness["faithfulness_score"] = (
    question_faithfulness["supported_claims"]
    / question_faithfulness["total_claims"]
)

# Add question-level faithfulness to main results
faithfulness_map = (
    question_faithfulness
    .set_index("question_id")["faithfulness_score"]
)

results_df["faithfulness_score"] = (
    results_df["question_id"]
    .map(faithfulness_map)
)

# Mean question-level faithfulness by type
faithfulness_by_type = (
    question_faithfulness
    .groupby("question_type")["faithfulness_score"]
    .mean()
    .round(3)
    .reset_index()
)

# Overall claim-level measures
overall_faithfulness = (
    claim_eval_df["supported"].sum()
    / len(claim_eval_df)
)

unsupported_claim_rate = (
    1 - overall_faithfulness
)

display(question_faithfulness)

display(faithfulness_by_type)

print(
    "Overall claim-level faithfulness:",
    round(overall_faithfulness, 3)
)

print(
    "Unsupported claim rate:",
    round(unsupported_claim_rate, 3)
)

,question_id,question_type,supported_claims,total_claims,faithfulness_score
0,I01,Inferred,3,5,0.6
1,I02,Inferred,4,4,1.0
2,I03,Inferred,6,6,1.0
3,I04,Inferred,3,3,1.0
4,K01,Known,1,1,1.0
5,K02,Known,1,1,1.0
6,K03,Known,2,2,1.0
7,K04,Known,1,1,1.0
8,K05,Known,3,3,1.0
9,K06,Known,1,1,1.0


,question_type,faithfulness_score
0,Inferred,0.9
1,Known,1.0


Overall claim-level faithfulness: 0.929
Unsupported claim rate: 0.071


## 6. Semantic Citation Evaluation

Citation quality is evaluated separately from simply checking whether a citation is present.

The same citation evaluation used for Version 1 is applied to Version 2:

- **Citation precision:** the proportion of cited chunk IDs that provide valid support for factual claims in the answer.
- **Citation coverage:** the proportion of supported factual claims that are backed by at least one appropriate citation.

Keeping the calculation unchanged allows citation behaviour to be compared directly between Version 1 and Version 2.

In [17]:
# Build the set of valid supporting chunks for each question

valid_supporting_chunks = (
    claim_eval_df[
        claim_eval_df["supported"] == 1
    ]
    .groupby("question_id")["supporting_chunk_ids"]
    .apply(
        lambda lists: set(
            chunk
            for chunk_list in lists
            for chunk in chunk_list
        )
    )
    .to_dict()
)


# Citation precision for each question
def calculate_citation_precision(row):
    cited = row["cited_chunk_ids"]

    if not cited:
        return np.nan

    valid_chunks = valid_supporting_chunks.get(
        row["question_id"],
        set()
    )

    correct_citations = sum(
        citation in valid_chunks
        for citation in cited
    )

    return correct_citations / len(cited)


results_df["citation_precision"] = results_df.apply(
    calculate_citation_precision,
    axis=1
)


# Check citation coverage at claim level
claim_eval_df["citation_covered"] = claim_eval_df.apply(
    lambda row:
        int(
            any(
                chunk in results_df.loc[
                    results_df["question_id"] == row["question_id"],
                    "cited_chunk_ids"
                ].iloc[0]
                for chunk in row["supporting_chunk_ids"]
            )
        )
        if row["supported"] == 1
        else np.nan,
    axis=1
)


# Citation coverage per question
citation_coverage = (
    claim_eval_df[
        claim_eval_df["supported"] == 1
    ]
    .groupby("question_id")["citation_covered"]
    .mean()
)

results_df["citation_coverage"] = (
    results_df["question_id"]
    .map(citation_coverage)
)


display(
    results_df[
        [
            "question_id",
            "question_type",
            "cited_chunk_ids",
            "citation_precision",
            "citation_coverage"
        ]
    ]
)


# Overall citation metrics
overall_citation_precision = (
    results_df.loc[
        results_df["citation_present"] == 1,
        "citation_precision"
    ].mean()
)

overall_citation_coverage = (
    claim_eval_df.loc[
        claim_eval_df["supported"] == 1,
        "citation_covered"
    ].mean()
)

print(
    "Overall semantic citation precision:",
    round(overall_citation_precision, 3)
)

print(
    "Overall supported-claim citation coverage:",
    round(overall_citation_coverage, 3)
)

,question_id,question_type,cited_chunk_ids,citation_precision,citation_coverage
0,K01,Known,[RG_P07],1.0,1.000000
1,K02,Known,[RG_P21],1.0,1.000000
2,K03,Known,"[MS_P01, RG_P39]",0.5,1.000000
3,K04,Known,[RG_P17],1.0,1.000000
4,K05,Known,[RG_P21],1.0,1.000000
5,K06,Known,[RG_P24],0.0,0.000000
6,K07,Known,[],NaN,NaN
7,K08,Known,[RG_P28],1.0,1.000000
8,I01,Inferred,[RG_P13],1.0,0.333333
9,I02,Inferred,[RG_P23],1.0,1.000000


Overall semantic citation precision: 0.864
Overall supported-claim citation coverage: 0.846


## 7. Answer Correctness

Each Version 2 response is manually reviewed for overall correctness using the same criteria as Version 1.

A response is marked as correct (`1`) when its main answer is factually appropriate and does not contain a major contradiction. An answer can still be incomplete while being marked correct because completeness is evaluated separately.

For Out-of-KB questions, a correct response requires the system to recognise that the retrieved evidence is insufficient and abstain from providing an unsupported answer.

In [18]:
# Human-reviewed Version 2 answer correctness

answer_correctness = {
    "K01": 1,
    "K02": 0,
    "K03": 1,
    "K04": 1,
    "K05": 1,
    "K06": 1,
    "K07": 0,
    "K08": 1,

    "I01": 0,
    "I02": 1,
    "I03": 1,
    "I04": 1,

    "O01": 1,
    "O02": 1,
    "O03": 1
}

results_df["correct"] = (
    results_df["question_id"]
    .map(answer_correctness)
)

correctness_by_type = (
    results_df
    .groupby("question_type")["correct"]
    .mean()
    .round(3)
    .reset_index()
)

overall_correctness = (
    results_df["correct"].mean()
)

display(
    results_df[
        [
            "question_id",
            "question_type",
            "correct",
            "completeness_score",
            "faithfulness_score"
        ]
    ]
)

display(correctness_by_type)

print(
    "Overall answer correctness:",
    round(overall_correctness, 3)
)

,question_id,question_type,correct,completeness_score,faithfulness_score
0,K01,Known,1,1.000000,1.0
1,K02,Known,0,0.000000,1.0
2,K03,Known,1,1.000000,1.0
3,K04,Known,1,1.000000,1.0
4,K05,Known,1,1.000000,1.0
5,K06,Known,1,1.000000,1.0
6,K07,Known,0,0.000000,NaN
7,K08,Known,1,0.500000,1.0
8,I01,Inferred,0,0.333333,0.6
9,I02,Inferred,1,0.500000,1.0


,question_type,correct
0,Inferred,0.75
1,Known,0.75
2,Out-of-KB,1.00


Overall answer correctness: 0.8


## 8. Version 2 Evaluation Summary

The Version 2 evaluation combines retrieval, answer quality, grounding, citation and Out-of-KB measures using the same evaluation framework as Version 1.

Retrieval remains unchanged between the two versions, while the remaining metrics measure the effect of the revised generation prompt.

This provides the complete Version 2 result set for the final A/B comparison with Version 1.

In [19]:
# Consolidated Version 2 evaluation metrics

supported_questions = results_df[
    results_df["question_type"].isin(["Known", "Inferred"])
]

summary_metrics = pd.DataFrame({
    "evaluation_area": [
        "Retrieval",
        "Retrieval",
        "Retrieval",
        "Answer quality",
        "Answer quality",
        "Answer quality",
        "Grounding",
        "Grounding",
        "Citations",
        "Citations",
        "Citations",
        "Citations",
        "Out-of-KB"
    ],

    "metric": [
        "Recall@5",
        "MRR@5",
        "NDCG@5",
        "Answer correctness",
        "Mean question completeness",
        "Gold-fact coverage",
        "Claim-level faithfulness",
        "Unsupported claim rate",
        "Citation presence",
        "Citations from retrieved context",
        "Semantic citation precision",
        "Supported-claim citation coverage",
        "Out-of-KB abstention accuracy"
    ],

    "score": [
        supported_questions["recall_at_k"].mean(),
        supported_questions["mrr_at_k"].mean(),
        supported_questions["ndcg_at_k"].mean(),

        results_df["correct"].mean(),
        question_completeness["completeness_score"].mean(),
        fact_eval_df["covered"].sum() / len(fact_eval_df),

        claim_eval_df["supported"].sum() / len(claim_eval_df),
        1 - (
            claim_eval_df["supported"].sum()
            / len(claim_eval_df)
        ),

        results_df["citation_present"].mean(),

        results_df.loc[
            results_df["citation_present"] == 1,
            "citations_from_retrieved"
        ].mean(),

        results_df.loc[
            results_df["citation_present"] == 1,
            "citation_precision"
        ].mean(),

        claim_eval_df.loc[
            claim_eval_df["supported"] == 1,
            "citation_covered"
        ].mean(),

        results_df.loc[
            results_df["question_type"] == "Out-of-KB",
            "abstention_correct"
        ].mean()
    ]
})

summary_metrics["score"] = (
    summary_metrics["score"]
    .astype(float)
    .round(3)
)

display(summary_metrics)

,evaluation_area,metric,score
0,Retrieval,Recall@5,0.979
1,Retrieval,MRR@5,0.808
2,Retrieval,NDCG@5,0.831
3,Answer quality,Answer correctness,0.800
4,Answer quality,Mean question completeness,0.594
5,Answer quality,Gold-fact coverage,0.515
6,Grounding,Claim-level faithfulness,0.929
7,Grounding,Unsupported claim rate,0.071
8,Citations,Citation presence,0.733
9,Citations,Citations from retrieved context,1.000


## 9. Evaluation Scope

This section summarises the size and structure of the Version 2 evaluation.

The knowledge base, test collection, retrieval method, generation model and top-k setting are kept consistent with Version 1. This ensures that the final comparison isolates the effect of the revised Version 2 generation prompt.

In [20]:
# Version 2 evaluation scope

evaluation_scope = pd.DataFrame({
    "item": [
        "System version",
        "Retriever",
        "Generation model",
        "Top-k",
        "Knowledge-base chunks",
        "Total test questions",
        "Known questions",
        "Inferred questions",
        "Out-of-KB questions",
        "Human-reviewed gold facts",
        "Human-reviewed generated claims"
    ],

    "value": [
        SYSTEM_VERSION,
        "BM25",
        OLLAMA_MODEL,
        TOP_K,
        len(kb_df),
        len(results_df),
        (results_df["question_type"] == "Known").sum(),
        (results_df["question_type"] == "Inferred").sum(),
        (results_df["question_type"] == "Out-of-KB").sum(),
        len(fact_eval_df),
        len(claim_eval_df)
    ]
})

display(evaluation_scope)

,item,value
0,System version,V2
1,Retriever,BM25
2,Generation model,llama3.2:3b
3,Top-k,5
4,Knowledge-base chunks,32
5,Total test questions,15
6,Known questions,8
7,Inferred questions,4
8,Out-of-KB questions,3
9,Human-reviewed gold facts,33


## 10. Performance by Question Type

Performance is summarised separately for Known, Inferred and Out-of-KB questions.

This helps show how Version 2 behaves across different question types rather than relying only on overall averages. The same breakdown used for Version 1 is retained for the final A/B comparison.

In [21]:
# Version 2 performance by question type

type_rows = []

for question_type, group in results_df.groupby(
    "question_type",
    sort=False
):
    type_rows.append({
        "question_type": question_type,
        "questions": len(group),

        "recall_at_5": (
            group["recall_at_k"].mean()
            if question_type != "Out-of-KB"
            else np.nan
        ),

        "answer_correctness": (
            group["correct"].mean()
        ),

        "mean_completeness": (
            group["completeness_score"].mean()
            if question_type != "Out-of-KB"
            else np.nan
        ),

        "mean_faithfulness": (
            group["faithfulness_score"].mean()
        ),

        "citation_presence": (
            group["citation_present"].mean()
        ),

        "abstention_accuracy": (
            group["abstention_correct"].mean()
            if question_type == "Out-of-KB"
            else np.nan
        )
    })


type_summary = pd.DataFrame(type_rows)

numeric_columns = [
    "recall_at_5",
    "answer_correctness",
    "mean_completeness",
    "mean_faithfulness",
    "citation_presence",
    "abstention_accuracy"
]

type_summary[numeric_columns] = (
    type_summary[numeric_columns]
    .round(3)
)

display(type_summary)

,question_type,questions,recall_at_5,answer_correctness,mean_completeness,mean_faithfulness,citation_presence,abstention_accuracy
0,Known,8,1.000,0.75,0.688,1.0,0.875,NaN
1,Inferred,4,0.938,0.75,0.408,0.9,1.000,NaN
2,Out-of-KB,3,NaN,1.00,NaN,NaN,0.000,1.0


## 11. Failure Analysis

Each Version 2 question is classified by the types of failure observed in the pipeline.

The same failure categories used for Version 1 are retained so changes in failure behaviour can be compared directly between the two versions.

In [22]:
def classify_failures(row):
    failures = []

    # Supported questions
    if row["question_type"] != "Out-of-KB":

        if (
            pd.notna(row["recall_at_k"])
            and row["recall_at_k"] < 1
        ):
            failures.append("Missing retrieval evidence")

        if row["correct"] == 0:
            failures.append("Incorrect answer")

        if (
            pd.notna(row["completeness_score"])
            and row["completeness_score"] < 1
        ):
            failures.append("Incomplete answer")

        if (
            pd.notna(row["faithfulness_score"])
            and row["faithfulness_score"] < 1
        ):
            failures.append("Unsupported claim(s)")

        citation_failure = (
            row["citation_present"] == 0
            or (
                pd.notna(row["citation_precision"])
                and row["citation_precision"] < 1
            )
            or (
                pd.notna(row["citation_coverage"])
                and row["citation_coverage"] < 1
            )
        )

        if citation_failure:
            failures.append("Citation failure")

    # Out-of-KB questions
    else:
        if row["abstention_correct"] == 0:
            failures.append("Failed abstention")

        if row["correct"] == 0:
            failures.append("Incorrect answer")

        if (
            pd.notna(row["faithfulness_score"])
            and row["faithfulness_score"] < 1
        ):
            failures.append("Unsupported claim(s)")

    if not failures:
        failures.append("No identified failure")

    return failures


results_df["failure_types"] = results_df.apply(
    classify_failures,
    axis=1
)

results_df["failure_count"] = (
    results_df["failure_types"]
    .apply(
        lambda x:
            0
            if x == ["No identified failure"]
            else len(x)
    )
)

display(
    results_df[
        [
            "question_id",
            "question_type",
            "failure_count",
            "failure_types"
        ]
    ]
)

,question_id,question_type,failure_count,failure_types
0,K01,Known,0,[No identified failure]
1,K02,Known,2,"[Incorrect answer, Incomplete answer]"
2,K03,Known,1,[Citation failure]
3,K04,Known,0,[No identified failure]
4,K05,Known,0,[No identified failure]
5,K06,Known,1,[Citation failure]
6,K07,Known,3,"[Incorrect answer, Incomplete answer, Citation..."
7,K08,Known,1,[Incomplete answer]
8,I01,Inferred,5,"[Missing retrieval evidence, Incorrect answer,..."
9,I02,Inferred,1,[Incomplete answer]


In [23]:
from collections import Counter

failure_counts = Counter(
    failure
    for failures in results_df["failure_types"]
    for failure in failures
    if failure != "No identified failure"
)

failure_summary = pd.DataFrame(
    [
        {
            "failure_type": failure,
            "questions_affected": count,
            "share_of_test_set": (
                count / len(results_df)
            )
        }
        for failure, count
        in failure_counts.items()
    ]
)

failure_summary = (
    failure_summary
    .sort_values(
        "questions_affected",
        ascending=False
    )
    .reset_index(drop=True)
)

failure_summary["share_of_test_set"] = (
    failure_summary["share_of_test_set"]
    .round(3)
)

display(failure_summary)

,failure_type,questions_affected,share_of_test_set
0,Incomplete answer,7,0.467
1,Citation failure,5,0.333
2,Incorrect answer,3,0.200
3,Missing retrieval evidence,1,0.067
4,Unsupported claim(s),1,0.067


### Failure Analysis Summary

Version 2 still shows **incomplete answers** as the most common failure, affecting 7 of 15 questions (46.7%).

**Citation failures** affected 5 questions (33.3%), while **incorrect answers** affected 3 questions (20.0%).

Only one question was affected by **missing retrieval evidence**, and one question contained **unsupported claims**.

No Out-of-KB questions were classified as failed abstentions in Version 2.

In [24]:
# Save Version 2 evaluation outputs

OUTPUT_DIR = DATA_DIR.parent / "evaluation"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

results_df.to_csv(
    OUTPUT_DIR / "v2_question_results.csv",
    index=False
)

fact_eval_df.to_csv(
    OUTPUT_DIR / "v2_gold_fact_evaluation.csv",
    index=False
)

claim_eval_df.to_csv(
    OUTPUT_DIR / "v2_claim_evaluation.csv",
    index=False
)

summary_metrics.to_csv(
    OUTPUT_DIR / "v2_summary_metrics.csv",
    index=False
)

evaluation_scope.to_csv(
    OUTPUT_DIR / "v2_evaluation_scope.csv",
    index=False
)

type_summary.to_csv(
    OUTPUT_DIR / "v2_question_type_summary.csv",
    index=False
)

failure_summary.to_csv(
    OUTPUT_DIR / "v2_failure_summary.csv",
    index=False
)

print("Saved Version 2 evaluation outputs to:")
print(OUTPUT_DIR)

Saved Version 2 evaluation outputs to:
/Users/seanrichards/Documents/University/DS Case Studies/Project/data/evaluation


## 12. Assignment Brief Specific Metrics

The assignment brief requires the evaluation framework to measure **effectiveness**, with the percentage of unanswered questions provided as one example.

For Version 2, effectiveness is reported using several related measures because a higher unanswered rate can be appropriate when the system correctly refuses questions that are not supported by the knowledge base.

The measures below therefore separate overall correctness, unanswered questions, coverage of answerable questions and correct Out-of-KB abstention.

In [25]:
# Assignment-specific effectiveness measures

unanswered_rate = results_df["abstained"].mean()

supported_mask = results_df["question_type"].isin(
    ["Known", "Inferred"]
)

answerable_question_coverage = (
    1 - results_df.loc[
        supported_mask,
        "abstained"
    ].mean()
)

out_of_kb_abstention_accuracy = (
    results_df.loc[
        results_df["question_type"] == "Out-of-KB",
        "abstention_correct"
    ].mean()
)

effectiveness_metrics = pd.DataFrame({
    "metric": [
        "Answer correctness",
        "Unanswered rate",
        "Answerable-question coverage",
        "Out-of-KB abstention accuracy"
    ],
    "score": [
        results_df["correct"].mean(),
        unanswered_rate,
        answerable_question_coverage,
        out_of_kb_abstention_accuracy
    ]
})

effectiveness_metrics["score"] = (
    effectiveness_metrics["score"]
    .astype(float)
    .round(3)
)

display(effectiveness_metrics)

,metric,score
0,Answer correctness,0.800
1,Unanswered rate,0.267
2,Answerable-question coverage,0.917
3,Out-of-KB abstention accuracy,1.000


## 13. Version 1 vs Version 2 Comparison

Version 1 and Version 2 are compared using the same knowledge base, test collection, BM25 retriever, top-k setting, generation model and evaluation framework.

The only intended system change is the revised Version 2 generation prompt. This makes the comparison a controlled A/B test of how the stronger grounding, citation and abstention instructions affected system behaviour.

The table below reports both versions side-by-side together with the numerical change from Version 1 to Version 2.

In [26]:
# Load saved Version 1 and Version 2 results

v1_summary = pd.read_csv(
    OUTPUT_DIR / "v1_summary_metrics.csv"
)

v2_summary = pd.read_csv(
    OUTPUT_DIR / "v2_summary_metrics.csv"
)

v1_questions = pd.read_csv(
    OUTPUT_DIR / "v1_question_results.csv"
)

v2_questions = pd.read_csv(
    OUTPUT_DIR / "v2_question_results.csv"
)


# Add effectiveness measures not included in the main summary tables
def calculate_effectiveness_metrics(df):
    supported_mask = df["question_type"].isin(
        ["Known", "Inferred"]
    )

    return {
        "Unanswered rate": df["abstained"].mean(),

        "Answerable-question coverage": (
            1 - df.loc[
                supported_mask,
                "abstained"
            ].mean()
        )
    }


v1_effectiveness = calculate_effectiveness_metrics(
    v1_questions
)

v2_effectiveness = calculate_effectiveness_metrics(
    v2_questions
)


# Convert summary tables to metric dictionaries
v1_scores = dict(
    zip(
        v1_summary["metric"],
        v1_summary["score"]
    )
)

v2_scores = dict(
    zip(
        v2_summary["metric"],
        v2_summary["score"]
    )
)

v1_scores.update(v1_effectiveness)
v2_scores.update(v2_effectiveness)


# Keep metrics in a consistent presentation order
comparison_metrics = [
    "Recall@5",
    "MRR@5",
    "NDCG@5",
    "Answer correctness",
    "Mean question completeness",
    "Gold-fact coverage",
    "Claim-level faithfulness",
    "Unsupported claim rate",
    "Citation presence",
    "Citations from retrieved context",
    "Semantic citation precision",
    "Supported-claim citation coverage",
    "Out-of-KB abstention accuracy",
    "Unanswered rate",
    "Answerable-question coverage"
]


comparison_df = pd.DataFrame({
    "metric": comparison_metrics,
    "V1": [
        v1_scores[metric]
        for metric in comparison_metrics
    ],
    "V2": [
        v2_scores[metric]
        for metric in comparison_metrics
    ]
})

comparison_df["change"] = (
    comparison_df["V2"]
    - comparison_df["V1"]
)

comparison_df[["V1", "V2", "change"]] = (
    comparison_df[
        ["V1", "V2", "change"]
    ]
    .astype(float)
    .round(3)
)

display(comparison_df)


# Save final A/B comparison
comparison_df.to_csv(
    OUTPUT_DIR / "v1_v2_summary_comparison.csv",
    index=False
)

print(
    "Saved V1 vs V2 comparison to:",
    OUTPUT_DIR / "v1_v2_summary_comparison.csv"
)

,metric,V1,V2,change
0,Recall@5,0.979,0.979,0.000
1,MRR@5,0.808,0.808,0.000
2,NDCG@5,0.831,0.831,0.000
3,Answer correctness,0.800,0.800,0.000
4,Mean question completeness,0.732,0.594,-0.138
5,Gold-fact coverage,0.606,0.515,-0.091
6,Claim-level faithfulness,0.800,0.929,0.129
7,Unsupported claim rate,0.200,0.071,-0.129
8,Citation presence,0.600,0.733,0.133
9,Citations from retrieved context,1.000,1.000,0.000


Saved V1 vs V2 comparison to: /Users/seanrichards/Documents/University/DS Case Studies/Project/data/evaluation/v1_v2_summary_comparison.csv


### A/B Comparison Summary

The controlled comparison shows several clear changes between Version 1 and Version 2:

- Retrieval performance is unchanged because both versions use the same BM25 pipeline and top-five retrieved chunks.
- Overall answer correctness remains at **0.800**.
- Claim-level faithfulness increases from **0.800 to 0.929**, while the unsupported claim rate decreases from **0.200 to 0.071**.
- Semantic citation precision increases from **0.778 to 0.864**, and supported-claim citation coverage increases from **0.600 to 0.846**.
- Out-of-KB abstention accuracy increases from **0.333 to 1.000**.
- Mean question completeness decreases from **0.732 to 0.594**, and answerable-question coverage decreases from **1.000 to 0.917**.
- The unanswered rate increases from **0.067 to 0.267**, reflecting the three correct Out-of-KB abstentions as well as one incorrect abstention on a supported question.

These results provide the quantitative basis for comparing the behaviour of the two generation approaches.